# ColdSite-DTI — DAVIS half of the 24-run grid (Kaggle, T4 x2)

Trains the **12 DAVIS cells** (4 splits x 3 seeds), regression, ColdSite-DTI.
The KIBA twelve run in `kaggle_kiba_grid.ipynb`. Together they are STATUS.md item 6.

## Before you run

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (Settings -> Accelerator) |
| Internet | **On** (Settings -> Internet) |
| Environment | **Pin to original** (Settings -> Environment Preferences) |
| How to run | **Save Version -> Save & Run All (Commit)** |

Not an interactive session: Kaggle idles those out after ~20 minutes and a
part-finished cell is lost.

## About the two cells already trained on Colab

`random/seed1` (CI 0.8234) and `random/seed2` (CI 0.8813) were trained on Colab and
live in Drive. **This notebook retrains them**, on purpose: Colab ran Python 3.13 with
torch 2.11.0+cu128 and Kaggle runs Python 3.12 with its own build, so importing those
two would leave the DAVIS half split across two library environments -- a difference
the paper would have to explain and could not quantify. Twelve cells in one environment
costs about 3.5 extra hours and removes the footnote.

To import them instead: upload the `.pt`, `*_results.json` and `*_history.json` files
from Drive as a Kaggle Dataset, attach it, and copy them into `RESULTS` before cell 6.
`run_grid` will then skip both.

## Why two processes

The model is single-GPU, so a second T4 idles unless something else uses it. This runs
**two `run_grid` processes**, one pinned per GPU, splitting the four split types 2/2.

| | GPU 0 | GPU 1 |
|---|---|---|
| splits | `random`, `cold_drug` | `cold_target`, `cold_pair` |
| cells | 6 | 6 |

## What to expect

Measured on Colab's T4: **~2.3 min/epoch**. Early stopping ended one cell at epoch 33
and another at 69, so cells run **~1.3-2.6 h** and six per GPU is **~8-15 h**. That may
not fit one 12-hour commit; finished cells are skipped on the next one.

**Resumable, and safely.** A cell interrupted by the 12-hour limit is *retrained* rather
than banked as finished (`d9a03c1`), and the end-to-end validation cell picks the first
*unfinished* cell rather than blindly the first one (`f9d36f0`) -- so resuming does not
start by retraining work that is already done. There is no within-cell resume, so a cell
cut off near the limit is lost.


## 1. Check the GPUs


In [ ]:
import torch

assert torch.cuda.is_available(), 'No CUDA. Settings -> Accelerator -> GPU T4 x2.'

N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
print('torch  :', torch.__version__)
print('devices:', N_GPU)

# batch 64 needs ~8.7 GB on a 1000-residue protein; a T4 has 15.6 GB.
BATCH = 64 if torch.cuda.get_device_properties(0).total_memory/1e9 >= 14 else 16
print('batch  :', BATCH)
if N_GPU < 2:
    print('\nOnly one GPU — the notebook will run a single process over all 12 cells.')


## 2. Clone the repo

Pinned to `main` on the fork, the same branch the KIBA half uses.


In [ ]:
import os, subprocess

REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
WORK = '/kaggle/working'
SRC  = f'{WORK}/ColdSite-DTI_New'

if not os.path.exists(SRC):
    !git clone --branch main {REPO} {SRC}
os.chdir(SRC)
!git pull origin main
!pip install -q tabulate subword-nmt

# Results live outside the clone so re-cloning never destroys finished cells.
RESULTS = f'{WORK}/results'
os.makedirs(RESULTS, exist_ok=True)
print()
!git log --oneline -1
print('results ->', RESULTS)
print('existing checkpoints:',
      len([f for f in os.listdir(RESULTS) if f.endswith('.pt')]), '/ 12')


## 3. Fetch the DeepDTA source files


In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'

for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}

!python -m src.data.load_data


Expected, exactly:

```
davis: 30056 measured pairs, 68 unique drugs, 442 unique targets, Y range [5.000, 10.796]
kiba: 118254 measured pairs, 2111 unique drugs, 229 unique targets, Y range [0.000, 17.200]
```


## 4. Build the splits — and verify they match

The DAVIS cells trained here must come from the same splits as the KIBA cells trained
in the other notebook, or the 24 are not one grid. These counts were verified on a
MacBook and on Colab; the cell asserts rather than trusting.


In [ ]:
!python -m src.data.build_splits

import pandas as pd

EXPECTED = {
    'random':      (21039, 3006, 6011),
    'cold_drug':   (21658, 2652, 5746),
    'cold_target': (21080, 2992, 5984),
    'cold_pair':   (15190,  264, 1144),
}

problems = []
for split, expected in EXPECTED.items():
    got = tuple(len(pd.read_csv(f'data/splits/davis/{split}/{part}.csv'))
                for part in ('train', 'valid', 'test'))
    if got != expected:
        problems.append(f'{split}: expected {expected}, got {got}')
    print(f"{split:12s} {str(got):26s} {'OK' if got == expected else 'MISMATCH'}")

assert not problems, (
    'DAVIS splits do not match the ones the rest of the grid was built from:\n  '
    + '\n  '.join(problems)
    + '\nDo not train on these.')
print('\ncold_pair validation really is only 264 rows — inherent to requiring both the'
      '\ndrug and the target to be unseen. Early stopping on it is noisy across seeds.')
print('\nSplits match. Safe to train.')


## 5. Preflight


In [ ]:
!python -m src.model.run_grid --preflight --datasets davis --results-dir {RESULTS} 2>&1 | tail -5


## 5b. Profile one forward pass (about a minute)

Runs while both GPUs are idle, so the numbers mean something.

`coldsite_dti.py` line 54 discards the protein encoder's self-attention weights, but the
encoder still asks for them — materialising a `(batch, len, len)` tensor and forcing
`nn.MultiheadAttention` off its fused kernel. The gap between the two `attention` rows is
what that costs. If `BiLSTM branch` dominates both, the ~700-1000 sequential timesteps are
the real cost and no attention change is worth making.


In [ ]:
!python -u -m src.model.profile_forward --batch-size {BATCH} --protein-len 1000


## 6. Train — two processes, one per GPU

Each process validates its own first unfinished cell end to end before launching the
rest. Logs are unbuffered and tailed every ~5 minutes.


In [ ]:
import subprocess, time, itertools

ASSIGNMENT = ([('0', 'random,cold_drug'), ('1', 'cold_target,cold_pair')]
              if N_GPU >= 2 else
              [('0', 'random,cold_drug,cold_target,cold_pair')])

procs = []
for gpu, splits in ASSIGNMENT:
    log = open(f'{WORK}/gpu{gpu}.log', 'w')
    # PYTHONUNBUFFERED is not optional: stdout is a file, so Python block-buffers
    # it and the epoch lines would sit unread for hours while the run looked hung.
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': gpu, 'PYTHONUNBUFFERED': '1'}
    cmd = ['python', '-u', '-m', 'src.model.run_grid',
           '--datasets', 'davis',
           '--splits', splits,
           '--batch-size', str(BATCH),
           '--results-dir', RESULTS]
    procs.append((gpu, subprocess.Popen(cmd, env=env, stdout=log,
                                        stderr=subprocess.STDOUT), log))
    print(f'GPU {gpu} -> {splits}')

print()
for tick in itertools.count():
    alive = [g for g, p, _ in procs if p.poll() is None]
    if not alive:
        break
    if tick % 10 == 0:            # every ~5 minutes
        for gpu, _, _ in procs:
            tail = subprocess.run(['tail', '-3', f'{WORK}/gpu{gpu}.log'],
                                  capture_output=True, text=True).stdout.strip()
            print(f'--- GPU {gpu} ---\n{tail}')
        done = len([f for f in os.listdir(RESULTS) if f.endswith('_results.json')])
        print(f'[{time.strftime("%H:%M:%S")}] complete: {done}/12   running: {alive}\n')
    time.sleep(30)

for gpu, p, log in procs:
    log.close()
    print(f'GPU {gpu} exited with code {p.returncode}')


## 7. What landed


In [ ]:
import glob, json as _json

ckpt = glob.glob(f'{RESULTS}/*davis*.pt')
res  = glob.glob(f'{RESULTS}/*davis*_results.json')
print('checkpoints :', len(ckpt), '/ 12')
print('run results :', len(res), '/ 12')

stems = {os.path.basename(p)[:-3] for p in ckpt}
done  = {os.path.basename(p).replace('_results.json', '') for p in res}
print('interrupted :', len(stems - done), '(retrained on the next commit)')
print()

for path in sorted(res):
    payload = _json.load(open(path))
    m = payload.get('test_metrics', {})
    print(f"  {payload.get('split'):12s} seed{payload.get('seed')}  "
          f"ci={m.get('ci', float('nan')):.4f}  mse={m.get('mse', float('nan')):.4f}")

print()
!python -m src.model.run_grid --preflight --datasets davis --results-dir {RESULTS} 2>&1 | tail -3


## 8. Getting the results back

Everything under `/kaggle/working` is saved as this version's output. Download `results/`
from the Output tab and put the `.pt` and `*_results.json` files in the same directory as
the KIBA ones — `run_faithfulness`, `run_ladder` and `run_audit` read all 24 cells from a
single directory and only work once both halves sit together.

Reference values from the Colab runs, for comparison against the reruns here:
`random/seed1` CI **0.8234** (best epoch 18, stopped 33), `random/seed2` CI **0.8813**
(best epoch 54, stopped 69). A different library environment will not reproduce these
exactly; a large gap is worth investigating, a small one is expected.
